# 04 - Model Evaluation & Validation: Making Sure Your Model Works

Welcome to Notebook 04! Now we'll learn how to:
1. Validate models properly with cross-validation
2. Understand confusion matrices and ROC curves
3. Prevent overfitting
4. Tune hyperparameters

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
from sklearn.datasets import make_classification

sns.set_style('whitegrid')
print("Libraries imported successfully!")

## Step 2: Create Sample Dataset

In [ ]:
# Create a more realistic dataset
X, y = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Class distribution (test set):")
print(f"  Class 0: {sum(y_test == 0)}")
print(f"  Class 1: {sum(y_test == 1)}")

## Step 3: Cross-Validation

**Why?** Get more reliable performance estimates!

In [ ]:
# Train a simple model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Cross-validation: Splits data into k folds
cv_scores = cross_val_score(
    model, X_train, y_train, cv=5, scoring='accuracy'
)

print("Cross-Validation Scores (5-fold):")
for fold, score in enumerate(cv_scores, 1):
    print(f"  Fold {fold}: {score:.4f}")

print(f"\nMean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print("\n💡 Cross-validation gives us confidence in model performance!")

## Step 4: Confusion Matrix

In [ ]:
# Train and predict
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Get confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print("\nInterpretation:")
print(f"  True Negatives (TN):  {cm[0, 0]} - Correctly predicted as Class 0")
print(f"  False Positives (FP): {cm[0, 1]} - Incorrectly predicted as Class 1")
print(f"  False Negatives (FN): {cm[1, 0]} - Incorrectly predicted as Class 0")
print(f"  True Positives (TP):  {cm[1, 1]} - Correctly predicted as Class 1")

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 5: Classification Report

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

print("\nMetric Explanations:")
print("""
Precision: Of positive predictions, how many were correct?
  Formula: TP / (TP + FP)
  Use when: False positives are expensive

Recall: Of actual positives, how many did we find?
  Formula: TP / (TP + FN)
  Use when: False negatives are expensive

F1-Score: Harmonic mean of precision and recall
  Formula: 2 * (Precision * Recall) / (Precision + Recall)
  Use when: You want balance between precision and recall

Support: Number of samples in each class
""")

## Step 6: ROC Curve & AUC

In [ ]:
# Get probability predictions
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

print(f"AUC Score: {roc_auc:.4f}")
print("\nAUC Interpretation:")
print("  0.5  = Random guessing")
print("  0.7+ = Good model")
print("  0.8+ = Very good model")
print("  0.9+ = Excellent model")
print(f"  1.0  = Perfect model")

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Overfitting Analysis

In [ ]:
# Train models with different complexity levels
max_depths = range(1, 21)
train_scores = []
test_scores = []

for depth in max_depths:
    model_temp = RandomForestClassifier(max_depth=depth, n_estimators=100, random_state=42)
    model_temp.fit(X_train, y_train)
    
    train_scores.append(model_temp.score(X_train, y_train))
    test_scores.append(model_temp.score(X_test, y_test))

# Plot
plt.figure(figsize=(10, 6))
plt.plot(max_depths, train_scores, 'o-', label='Training Score', linewidth=2, markersize=6)
plt.plot(max_depths, test_scores, 'o-', label='Testing Score', linewidth=2, markersize=6)
plt.xlabel('Tree Depth')
plt.ylabel('Accuracy')
plt.title('Overfitting Analysis: Training vs Testing Score')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Identify optimal depth
optimal_depth = max_depths[np.argmax(test_scores)]
print(f"\nOptimal tree depth: {optimal_depth}")
print(f"Training score at optimal depth: {train_scores[optimal_depth-1]:.4f}")
print(f"Testing score at optimal depth: {test_scores[optimal_depth-1]:.4f}")

print("\n📊 Analysis:")
if train_scores[optimal_depth-1] - test_scores[optimal_depth-1] > 0.1:
    print("⚠️  The model shows signs of overfitting")
    print("Consider: Reduce model complexity, use regularization, or get more data")
else:
    print("✅ The model generalizes well!")

## Step 8: Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10]
}

print("Testing", np.prod([len(v) for v in param_grid.values()]), "different combinations...")
print("This may take a moment...\n")

# Grid search
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n" + "="*50)
print("GRID SEARCH RESULTS")
print("="*50)
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")
print(f"Test Score with Best Model: {grid_search.score(X_test, y_test):.4f}")

In [ ]:
# Show top 10 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 'mean_test_score']]
results_df = results_df.sort_values('mean_test_score', ascending=False)

print("\nTop 10 Parameter Combinations:")
print(results_df.head(10).to_string(index=False))

## Step 9: Feature Importance

In [ ]:
# Get feature importance from best model
best_model = grid_search.best_estimator_
feature_importance = best_model.feature_importances_

# Create dataframe
feature_names = [f'Feature_{i}' for i in range(len(feature_importance))]
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("Feature Importance:")
print(importance_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

print("\n💡 Feature importance helps you understand which features matter most!")

## Step 10: Summary & Best Practices

In [ ]:
print("""
╔════════════════════════════════════════════════════════════╗
║           ML MODEL EVALUATION BEST PRACTICES               ║
╚════════════════════════════════════════════════════════════╝

1. ALWAYS USE CROSS-VALIDATION
   - Gives more reliable performance estimates
   - Use 5-fold or 10-fold cross-validation

2. EVALUATE ON TEST SET ONLY
   - Never evaluate on training data
   - Keep test set completely separate

3. USE APPROPRIATE METRICS
   - Classification: Accuracy, Precision, Recall, F1, AUC-ROC
   - Regression: MAE, RMSE, R²
   - Consider class imbalance

4. WATCH FOR OVERFITTING
   - High training accuracy but low test accuracy = Overfitting
   - Solutions:
     * Reduce model complexity
     * Use regularization
     * Get more data
     * Use cross-validation

5. HYPERPARAMETER TUNING
   - Use GridSearchCV or RandomizedSearchCV
   - Tune on training data using cross-validation
   - Evaluate final model on test set

6. UNDERSTAND YOUR METRICS
   - Precision matters when false positives are costly
   - Recall matters when false negatives are costly
   - AUC-ROC is good for imbalanced datasets

7. FEATURE IMPORTANCE
   - Understand which features drive predictions
   - Can help with feature selection
   - Improves model interpretability

8. DOCUMENT YOUR PROCESS
   - Keep track of experiments
   - Document hyperparameter choices
   - Explain your results
""")

## Congratulations! 🎉

You've completed the ML Beginner Project! You now know:
- ✅ How to explore and understand data
- ✅ How to clean and preprocess data
- ✅ How to build classification and regression models
- ✅ How to evaluate models properly
- ✅ How to prevent overfitting
- ✅ How to tune hyperparameters

## Next Steps:
1. **Practice**: Build projects with real datasets from Kaggle
2. **Learn**: Explore advanced algorithms (SVM, Neural Networks, etc.)
3. **Compete**: Join Kaggle competitions to test your skills
4. **Share**: Share your projects with the community

## Resources:
- Kaggle Datasets: https://www.kaggle.com/datasets
- UCI ML Repository: https://archive.ics.uci.edu/ml/
- Google Colab: https://colab.research.google.com/ (free GPU!)

**Keep learning and happy coding! 🚀**